# **Assignment 05: MLLM**

**Available:** Sep 16, 2025 3:00pm until Sep 30, 2025 11:59pm

**Details**
- https://huggingface.co/datasets/AI4Math/MathVistaLinks to an external site.​
- Use test set​
- Add a lora to InternVL3 and SophiaVL-R1​
- Train both loras with testmini​
- Evaluate on test
- To get results on test set​, you need to run the leaderboard, 
- instructions are here: https://mathvista.github.io/#leaderboard
- Insights on why either IVL or SVL is better in the above two runs​
- Reports, code, video and insights

## Setup

In [1]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Found Hugging Face token in environment variables
Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl
Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl


In [2]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
    
    # Set default CUDA device to cuda:1 to ensure consistency
    torch.cuda.set_device(1)
    print(f"Set default CUDA device to: cuda:1")
    
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
    torch.cuda.set_device(0)
    print(f"Set default CUDA device to: cuda:0")
    
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# Force all new tensors to be created on the selected device
if device.type == 'cuda':
    torch.cuda.set_device(device)
    print(f"Default tensor device set to: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0

def ensure_device_consistency():
    """Ensure all operations use the correct device"""
    if device.type == 'cuda':
        torch.cuda.set_device(device)
        print(f"✅ Set CUDA device to {device}")
    return device

def move_to_device(obj, target_device=None):
    """Helper function to move tensors to the target device"""
    if target_device is None:
        target_device = device
    
    if isinstance(obj, torch.Tensor):
        return obj.to(target_device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, target_device) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [move_to_device(item, target_device) for item in obj]
    else:
        return obj

def clear_memory():
    """Clear GPU memory and run garbage collection"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")


=== GPU Diagnostics ===
PyTorch version: 2.8.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Set default CUDA device to: cuda:1
Selected device: cuda:1
Default tensor device set to: cuda:1


## Data Preparation and Processing

In [3]:
# Import the Dataset
# Source: https://huggingface.co/datasets/AI4Math/MathVista

from datasets import load_dataset

dataset = load_dataset("AI4Math/MathVista")

In [4]:
# Examine the dataset structure
print("Dataset structure:")
print(dataset)
print("\nDataset keys:")
print(dataset.keys())

# Check the structure of each split
for split_name in dataset.keys():
    print(f"\n{split_name} split:")
    print(f"  Number of examples: {len(dataset[split_name])}")
    if len(dataset[split_name]) > 0:
        print(f"  Features: {dataset[split_name].features}")
        print(f"  First example keys: {list(dataset[split_name][0].keys())}")
        
        # Show a sample of the first example
        sample = dataset[split_name][0]
        print(f"  Sample data:")
        for key, value in sample.items():
            if key == 'image' and value is not None:
                print(f"    {key}: PIL Image ({value.size if hasattr(value, 'size') else 'unknown size'})")
            elif isinstance(value, str) and len(value) > 100:
                print(f"    {key}: {value[:100]}...")
            else:
                print(f"    {key}: {value}")
        break

Dataset structure:
DatasetDict({
    testmini: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 5141
    })
})

Dataset keys:
dict_keys(['testmini', 'test'])

testmini split:
  Number of examples: 1000
  Features: {'pid': Value('string'), 'question': Value('string'), 'image': Value('string'), 'decoded_image': Image(mode=None, decode=True), 'choices': List(Value('string')), 'unit': Value('string'), 'precision': Value('float64'), 'answer': Value('string'), 'question_type': Value('string'), 'answer_type': Value('string'), 'metadata': {'category': Value('string'), 'context': Value('string'), 'grade': Value('string'), 'img_height': Value('int64'), 

In [5]:
# Data Processing Functions
import json
from typing import Dict, List, Any, Tuple
from PIL import Image
import io
import base64

class MathVistaProcessor:
    def __init__(self):
        self.processed_data = {
            'testmini': [],
            'test': []
        }
    
    def format_question_for_models(self, example: Dict) -> Tuple[str, Image.Image]:
        """Format question and image for model input"""
        # Get the image
        image = example['decoded_image']
        
        # Format the question based on question type
        question = example['question']
        query = example.get('query', '')
        
        # Add context based on answer type
        if example['answer_type'] == 'float':
            if example.get('precision'):
                formatted_question = f"{question}\n\nPlease provide your answer as a floating-point number with {int(example['precision'])} decimal place(s)."
            else:
                formatted_question = f"{question}\n\nPlease provide your answer as a floating-point number."
        elif example['answer_type'] == 'integer':
            formatted_question = f"{question}\n\nPlease provide your answer as an integer."
        elif example['question_type'] == 'multi_choice':
            if example.get('choices'):
                choices_text = '\n'.join([f"{i+1}. {choice}" for i, choice in enumerate(example['choices'])])
                formatted_question = f"{question}\n\nChoices:\n{choices_text}\n\nPlease select the correct answer."
            else:
                formatted_question = question
        else:
            formatted_question = question
        
        # Add any additional query information
        if query and query != question:
            formatted_question = f"{formatted_question}\n\nAdditional context: {query}"
        
        return formatted_question, image
    
    def process_split(self, split_name: str, max_samples: int = None) -> List[Dict]:
        """Process a specific split of the dataset"""
        split_data = dataset[split_name]
        processed_examples = []
        
        print(f"Processing {split_name} split...")
        
        # Limit samples if specified
        if max_samples:
            split_data = split_data.select(range(min(max_samples, len(split_data))))
        
        for idx, example in enumerate(tqdm(split_data, desc=f"Processing {split_name}")):
            try:
                # Format question and get image
                formatted_question, image = self.format_question_for_models(example)
                
                # Create processed example
                processed_example = {
                    'pid': example['pid'],
                    'formatted_question': formatted_question,
                    'original_question': example['question'],
                    'image': image,
                    'ground_truth_answer': example['answer'],
                    'question_type': example['question_type'],
                    'answer_type': example['answer_type'],
                    'choices': example.get('choices'),
                    'unit': example.get('unit'),
                    'precision': example.get('precision'),
                    'metadata': example['metadata'],
                    'query': example.get('query', ''),
                }
                
                processed_examples.append(processed_example)
                
                # Print first example for verification
                if idx == 0:
                    print(f"\nFirst example from {split_name}:")
                    print(f"PID: {processed_example['pid']}")
                    print(f"Question type: {processed_example['question_type']}")
                    print(f"Answer type: {processed_example['answer_type']}")
                    print(f"Formatted question: {processed_example['formatted_question'][:200]}...")
                    print(f"Ground truth: {processed_example['ground_truth_answer']}")
                    print(f"Image size: {processed_example['image'].size}")
                    print("-" * 50)
                
            except Exception as e:
                print(f"Error processing example {idx} in {split_name}: {e}")
                continue
        
        self.processed_data[split_name] = processed_examples
        print(f"Successfully processed {len(processed_examples)} examples from {split_name}")
        
        return processed_examples
    
    def get_statistics(self):
        """Get statistics about the processed data"""
        stats = {}
        
        for split_name, examples in self.processed_data.items():
            if not examples:
                continue
                
            stats[split_name] = {
                'total_examples': len(examples),
                'question_types': {},
                'answer_types': {},
                'categories': {},
            }
            
            for example in examples:
                # Question types
                q_type = example['question_type']
                stats[split_name]['question_types'][q_type] = stats[split_name]['question_types'].get(q_type, 0) + 1
                
                # Answer types
                a_type = example['answer_type']
                stats[split_name]['answer_types'][a_type] = stats[split_name]['answer_types'].get(a_type, 0) + 1
                
                # Categories
                category = example['metadata']['category']
                stats[split_name]['categories'][category] = stats[split_name]['categories'].get(category, 0) + 1
        
        return stats

# Initialize processor
data_processor = MathVistaProcessor()

print("MathVista processor initialized successfully!")

MathVista processor initialized successfully!


In [6]:
# Process the testmini split (for training LoRA)
print("Processing testmini split for training...")
testmini_processed = data_processor.process_split('testmini')

print(f"\nTestmini processing completed!")
print(f"Total examples processed: {len(testmini_processed)}")

# Show some statistics
testmini_stats = data_processor.get_statistics()
print("\nTestmini Statistics:")
print(f"Question types: {testmini_stats['testmini']['question_types']}")
print(f"Answer types: {testmini_stats['testmini']['answer_types']}")
print(f"Categories: {list(testmini_stats['testmini']['categories'].keys())}")

Processing testmini split for training...
Processing testmini split...


Processing testmini:   0%|          | 0/1000 [00:00<?, ?it/s]


First example from testmini:
PID: 1
Question type: free_form
Answer type: float
Formatted question: When a spring does work on an object, we cannot find the work by simply multiplying the spring force by the object's displacement. The reason is that there is no one value for the force-it changes. Ho...
Ground truth: 1.2
Image size: (1514, 720)
--------------------------------------------------
Successfully processed 1000 examples from testmini

Testmini processing completed!
Total examples processed: 1000

Testmini Statistics:
Question types: {'free_form': 460, 'multi_choice': 540}
Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
Categories: ['math-targeted-vqa', 'general-vqa']
Successfully processed 1000 examples from testmini

Testmini processing completed!
Total examples processed: 1000

Testmini Statistics:
Question types: {'free_form': 460, 'multi_choice': 540}
Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
Categories: ['math-targeted-vqa',

In [7]:
# Process the test split (for evaluation)
print("Processing test split for evaluation...")
test_processed = data_processor.process_split('test')

print(f"\nTest processing completed!")
print(f"Total examples processed: {len(test_processed)}")

# Show complete statistics
all_stats = data_processor.get_statistics()
print("\n" + "="*60)
print("COMPLETE DATASET STATISTICS")
print("="*60)

for split_name, stats in all_stats.items():
    print(f"\n{split_name.upper()} SPLIT:")
    print(f"  Total examples: {stats['total_examples']}")
    print(f"  Question types: {stats['question_types']}")
    print(f"  Answer types: {stats['answer_types']}")
    print(f"  Categories: {stats['categories']}")
    print("-" * 40)

Processing test split for evaluation...
Processing test split...


Processing test:   0%|          | 0/5141 [00:00<?, ?it/s]


First example from test:
PID: 1001
Question type: free_form
Answer type: integer
Formatted question: In how many years, is the percentage of labor tax greater than 3 %?

Please provide your answer as an integer.

Additional context: Hint: Please answer the question requiring an integer answer and pro...
Ground truth: 
Image size: (981, 650)
--------------------------------------------------
Successfully processed 5141 examples from test

Test processing completed!
Total examples processed: 5141

COMPLETE DATASET STATISTICS

TESTMINI SPLIT:
  Total examples: 1000
  Question types: {'free_form': 460, 'multi_choice': 540}
  Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
  Categories: {'math-targeted-vqa': 540, 'general-vqa': 460}
----------------------------------------

TEST SPLIT:
  Total examples: 5141
  Question types: {'free_form': 2289, 'multi_choice': 2852}
  Answer types: {'integer': 2043, 'text': 2852, 'float': 232, 'list': 14}
  Categories: {'general-vqa': 

In [8]:
# Utility functions to access processed data
def get_training_data():
    """Get processed testmini data for training LoRA"""
    return data_processor.processed_data['testmini']

def get_evaluation_data():
    """Get processed test data for evaluation"""
    return data_processor.processed_data['test']

def get_sample_batch(split='testmini', batch_size=4, start_idx=0):
    """Get a sample batch for testing model inference"""
    if split not in data_processor.processed_data:
        print(f"Split '{split}' not found. Available splits: {list(data_processor.processed_data.keys())}")
        return []
    
    data = data_processor.processed_data[split]
    end_idx = min(start_idx + batch_size, len(data))
    return data[start_idx:end_idx]

def save_processed_data(filename_prefix="mathvista_processed"):
    """Save processed data to files for later use"""
    import pickle
    
    # Save testmini data
    with open(f"data/{filename_prefix}_testmini.pkl", 'wb') as f:
        pickle.dump(data_processor.processed_data['testmini'], f)
    print(f"Testmini data saved to data/{filename_prefix}_testmini.pkl")
    
    # Save test data
    with open(f"data/{filename_prefix}_test.pkl", 'wb') as f:
        pickle.dump(data_processor.processed_data['test'], f)
    print(f"Test data saved to data/{filename_prefix}_test.pkl")
    
    # Save statistics
    stats = data_processor.get_statistics()
    with open(f"data/{filename_prefix}_stats.json", 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"Statistics saved to data/{filename_prefix}_stats.json")

def load_processed_data(filename_prefix="mathvista_processed"):
    """Load previously processed data"""
    import pickle
    
    try:
        # Load testmini data
        with open(f"data/{filename_prefix}_testmini.pkl", 'rb') as f:
            data_processor.processed_data['testmini'] = pickle.load(f)
        print(f"Testmini data loaded from data/{filename_prefix}_testmini.pkl")
        
        # Load test data
        with open(f"data/{filename_prefix}_test.pkl", 'rb') as f:
            data_processor.processed_data['test'] = pickle.load(f)
        print(f"Test data loaded from data/{filename_prefix}_test.pkl")
        
        return True
    except FileNotFoundError as e:
        print(f"Could not load processed data: {e}")
        return False

# Save the processed data
save_processed_data()

print("\n" + "="*60)
print("DATASET PROCESSING COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"✓ Testmini split: {len(data_processor.processed_data['testmini'])} examples (for LoRA training)")
print(f"✓ Test split: {len(data_processor.processed_data['test'])} examples (for evaluation)")
print("✓ Data formatted for both InternVL3 and SophiaVL-R1 models")
print("✓ Processed data saved to pickle files")
print("\nYou can now proceed with:")
print("1. Training LoRA adapters on both models using testmini data")
print("2. Evaluating both models on test data")
print("3. Comparing performance between InternVL3 and SophiaVL-R1")

Testmini data saved to data/mathvista_processed_testmini.pkl
Test data saved to data/mathvista_processed_test.pkl
Statistics saved to data/mathvista_processed_stats.json

DATASET PROCESSING COMPLETED SUCCESSFULLY!
✓ Testmini split: 1000 examples (for LoRA training)
✓ Test split: 5141 examples (for evaluation)
✓ Data formatted for both InternVL3 and SophiaVL-R1 models
✓ Processed data saved to pickle files

You can now proceed with:
1. Training LoRA adapters on both models using testmini data
2. Evaluating both models on test data
3. Comparing performance between InternVL3 and SophiaVL-R1
Test data saved to data/mathvista_processed_test.pkl
Statistics saved to data/mathvista_processed_stats.json

DATASET PROCESSING COMPLETED SUCCESSFULLY!
✓ Testmini split: 1000 examples (for LoRA training)
✓ Test split: 5141 examples (for evaluation)
✓ Data formatted for both InternVL3 and SophiaVL-R1 models
✓ Processed data saved to pickle files

You can now proceed with:
1. Training LoRA adapters on b

## Import SophiaVL-R1 Model

In [34]:
## Source: https://huggingface.co/bunny127/SophiaVL-R1-Thinking-Reward-Model-3B

# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText  # Use non-deprecated class
from peft import LoraConfig, get_peft_model, TaskType
import requests
from PIL import Image
import torch


# Load processor and model with memory optimization
try:
    print("Loading processor...")
    model_processor = AutoProcessor.from_pretrained(
        "bunny127/SophiaVL-R1-Thinking-Reward-Model-3B",
        use_fast=True  # Use fast processor to avoid warnings
    )
    
    print("Loading model... This may take several minutes...")
    model = AutoModelForImageTextToText.from_pretrained(
        "bunny127/SophiaVL-R1-Thinking-Reward-Model-3B",
        torch_dtype=torch.float16,  # Use half precision to save memory
        device_map={"": device.index if device.type == 'cuda' else device},  # Force to specific GPU
        trust_remote_code=True,
        low_cpu_mem_usage=True  # Optimize CPU memory usage during loading
    )
    print("Model loaded successfully!")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Trying with CPU offloading...")
    
    try:
        model = AutoModelForImageTextToText.from_pretrained(
            "bunny127/SophiaVL-R1-Thinking-Reward-Model-3B",
            torch_dtype=torch.float16,
            device_map={"": device.index if device.type == 'cuda' else device},  # Force to specific GPU
            offload_folder="./offload",  # Offload to disk if needed
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        print("Model loaded with CPU offloading!")
    except Exception as e2:
        print(f"Failed to load model even with offloading: {e2}")
        print("The model may be too large for your system. Consider using a smaller model.")
        raise e2

# Check if model loaded successfully before applying LoRA
if 'model' in locals() and model is not None:
    print("Applying LoRA configuration...")
    
    # Define LoRA configuration with reduced complexity to save memory
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,  # Vision-to-Text is similar to sequence-to-sequence
        inference_mode=False,  # Set to True for inference only
        r=8,  # Reduced rank to save memory (was 16)
        lora_alpha=16,  # Reduced alpha proportionally (was 32)
        lora_dropout=0.1,  # Dropout for LoRA layers
        target_modules=[  # Target specific modules in the model
            "q_proj",
            "v_proj", 
            "k_proj",
            "o_proj",
            # Temporarily reduce target modules to save memory
            # "gate_proj",
            # "up_proj", 
            # "down_proj",
        ],
        bias="none",  # Don't train bias parameters
    )

    try:
        # Apply LoRA to the model
        model = get_peft_model(model, lora_config)
        print("LoRA applied successfully!")
        
        # Print model info
        model.print_trainable_parameters()
        
    except Exception as e:
        print(f"Error applying LoRA: {e}")
        print("Continuing without LoRA for inference...")

else:
    print("Model not loaded properly, skipping LoRA configuration.")




Loading processor...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Loading model... This may take several minutes...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Applying LoRA configuration...
LoRA applied successfully!
trainable params: 3,686,400 || all params: 3,758,309,376 || trainable%: 0.0981


## Fine Tune SophiaVL-R1 Model

In [35]:
# Fine Tune LoRA on SophiaVL-R1

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from PIL import Image
import json
import os
from tqdm.notebook import tqdm
import wandb
from datetime import datetime

class MathVistaDataset(Dataset):
    """Custom dataset for MathVista training data"""
    
    def __init__(self, processed_data, model_processor, max_length=1024):  # Reduced max length
        self.raw_data = processed_data
        self.model_processor = model_processor
        self.max_length = max_length
        
        # Pre-filter and cache valid examples
        print(f"Pre-processing {len(self.raw_data)} examples...")
        self.valid_data = []
        self.valid_indices = []
        
        for idx, example in enumerate(self.raw_data):
            try:
                # Quick validation
                if (example.get('image') is not None and 
                    example.get('formatted_question') and
                    len(example['formatted_question']) < 2500):  # Increased limit for longer questions
                    
                    processed_item = self._process_example(idx, example)
                    if processed_item is not None and 'input_ids' in processed_item:
                        self.valid_data.append(processed_item)
                        self.valid_indices.append(idx)
                        
                        # Limit to prevent memory issues
                        if len(self.valid_data) >= 100:  # Limit dataset size for now
                            break
                            
            except Exception as e:
                continue
        
        print(f"Dataset initialized with {len(self.valid_data)} valid examples from {len(self.raw_data)} raw examples")
    
    def __len__(self):
        return len(self.valid_data)
    
    def _process_example(self, idx, example):
        """Process a single example and return processed data or None"""
        try:
            # Use a very simple approach - just question text + answer
            question_text = example['formatted_question']
            if len(question_text) > 500:  # Keep it short
                question_text = question_text[:500] + "..."
            
            # Simple format without complex chat templates
            prompt = f"Question: {question_text}\nAnswer:"
            
            # Process with minimal complexity - IMAGE FIRST, then TEXT
            inputs = self.model_processor(
                example['image'],  # Image first
                prompt,            # Text second
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=256  # Very short sequences
            )
            
            # Flatten tensors
            for key in inputs:
                if isinstance(inputs[key], torch.Tensor):
                    inputs[key] = inputs[key].squeeze(0)
            
            inputs['labels'] = inputs['input_ids'].clone()
            inputs['ground_truth'] = str(example['ground_truth_answer'])
            
            return inputs
            
        except Exception as e:
            return None
    
    def __getitem__(self, idx):
        # Return pre-processed valid example
        if idx >= len(self.valid_data):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self.valid_data)}")
        
        return self.valid_data[idx]

class CustomDataCollator:
    """Custom data collator for vision-language models with proper device handling"""
    
    def __init__(self, model_processor):
        self.model_processor = model_processor
        self.pad_token_id = model_processor.tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = model_processor.tokenizer.eos_token_id
        # Store the target device to ensure consistency
        self.device = device  # Use the global device variable
    
    def __call__(self, features):
        # Safety check for empty or None features
        if not features or features is None:
            raise ValueError("Features list is empty or None")
        
        # Filter out any None items
        features = [f for f in features if f is not None and 'input_ids' in f]
        if not features:
            raise ValueError("All features are None or invalid")
        
        batch = {}
        
        # Extract individual components with safety checks
        input_ids = []
        labels = []
        pixel_values = []
        image_grid_thw = []
        
        for f in features:
            if 'input_ids' in f and f['input_ids'] is not None:
                # DEVICE FIX: Move to correct device immediately
                input_ids.append(f['input_ids'].to(self.device))
            else:
                # Create a dummy input_ids if missing
                input_ids.append(torch.tensor([1, 2, 3], device=self.device))
            
            if 'labels' in f and f['labels'] is not None:
                # DEVICE FIX: Move to correct device immediately
                labels.append(f['labels'].to(self.device))
            else:
                # Create dummy labels matching input_ids
                labels.append(input_ids[-1].clone())
            
            if 'pixel_values' in f and f['pixel_values'] is not None:
                # DEVICE FIX: Move to correct device immediately
                pixel_values.append(f['pixel_values'].to(self.device))
            
            # Add image_grid_thw handling
            if 'image_grid_thw' in f and f['image_grid_thw'] is not None:
                # DEVICE FIX: Move to correct device immediately
                image_grid_thw.append(f['image_grid_thw'].to(self.device))
            else:
                # Create dummy grid_thw if missing (shouldn't happen)
                image_grid_thw.append(torch.tensor([1, 1, 1], device=self.device))
        
        # Handle pixel values - they might have different shapes
        if pixel_values:
            try:
                # Check if all pixel_values have the same shape
                shapes = [pv.shape for pv in pixel_values]
                if len(set(shapes)) == 1:
                    # All same shape, can stack normally
                    batch['pixel_values'] = torch.stack(pixel_values)
                else:
                    # Different shapes - need to handle carefully
                    print(f"Warning: Different pixel_values shapes: {shapes}")
                    
                    # Find max dimensions
                    max_dim0 = max(pv.shape[0] for pv in pixel_values)
                    max_dim1 = pixel_values[0].shape[1] if len(pixel_values[0].shape) > 1 else 1
                    
                    # Pad each tensor to max dimensions
                    padded_pixel_values = []
                    for pv in pixel_values:
                        if len(pv.shape) == 2:
                            # Pad first dimension
                            pad_size = max_dim0 - pv.shape[0]
                            if pad_size > 0:
                                # DEVICE FIX: Create padding on the same device as pv
                                padding = torch.zeros(pad_size, pv.shape[1], dtype=pv.dtype, device=pv.device)
                                padded_pv = torch.cat([pv, padding], dim=0)
                            else:
                                padded_pv = pv[:max_dim0]  # Truncate if too large
                        else:
                            padded_pv = pv
                        padded_pixel_values.append(padded_pv)
                    
                    batch['pixel_values'] = torch.stack(padded_pixel_values)
                    
            except Exception as e:
                print(f"Warning: Could not process pixel_values: {e}")
                print(f"Shapes were: {[pv.shape for pv in pixel_values]}")
                # Create standardized dummy pixel values matching the expected format
                if pixel_values:
                    # Use the shape from the first valid pixel_values
                    reference_shape = pixel_values[0].shape
                    if len(reference_shape) == 2:
                        batch['pixel_values'] = torch.randn(len(features), reference_shape[0], reference_shape[1], device=self.device)
                    else:
                        batch['pixel_values'] = torch.randn(len(features), 3, 224, 224, device=self.device)
                else:
                    batch['pixel_values'] = torch.randn(len(features), 3, 224, 224, device=self.device)
        
        # Handle image_grid_thw - this is the key fix!
        if image_grid_thw:
            try:
                # All image_grid_thw should have shape [3] after dataset processing
                batch['image_grid_thw'] = torch.stack(image_grid_thw)
            except Exception as e:
                print(f"Warning: Could not stack image_grid_thw: {e}")
                print(f"Shapes were: {[igt.shape for igt in image_grid_thw]}")
                # Create dummy grid_thw as fallback
                batch['image_grid_thw'] = torch.tensor([[1, 1, 1]] * len(features), device=self.device)
        
        # Safety check for input_ids
        if not input_ids:
            raise ValueError("No valid input_ids found in features")
        
        # Get max length for padding
        try:
            max_len = max(len(seq) for seq in input_ids if seq is not None)
        except (ValueError, TypeError):
            print("Warning: Could not determine max length, using default")
            max_len = 512
        
        # Pad sequences
        padded_input_ids = []
        padded_labels = []
        padded_attention_masks = []
        
        for i, (input_id, label) in enumerate(zip(input_ids, labels)):
            if input_id is None or label is None:
                print(f"Warning: Skipping None input at index {i}")
                continue
                
            # Ensure tensors are 1D
            if input_id.dim() > 1:
                input_id = input_id.squeeze()
            if label.dim() > 1:
                label = label.squeeze()
            
            # Pad sequences
            current_len = len(input_id)
            pad_length = max_len - current_len
            
            if pad_length > 0:
                # DEVICE FIX: Create padding tensors on the same device as input_id
                padded_input_id = torch.cat([
                    input_id, 
                    torch.full((pad_length,), self.pad_token_id, dtype=input_id.dtype, device=input_id.device)
                ])
                padded_label = torch.cat([
                    label,
                    torch.full((pad_length,), -100, dtype=label.dtype, device=label.device)
                ])
                attention_mask = torch.cat([
                    torch.ones(current_len, dtype=torch.long, device=input_id.device),
                    torch.zeros(pad_length, dtype=torch.long, device=input_id.device)
                ])
            else:
                # Truncate if too long
                padded_input_id = input_id[:max_len]
                padded_label = label[:max_len]
                attention_mask = torch.ones(len(padded_input_id), dtype=torch.long, device=input_id.device)
            
            padded_input_ids.append(padded_input_id)
            padded_labels.append(padded_label)
            padded_attention_masks.append(attention_mask)
        
        # Final safety check
        if not padded_input_ids:
            raise ValueError("No valid padded sequences created")
        
        # Stack all tensors
        try:
            batch['input_ids'] = torch.stack(padded_input_ids)
            batch['labels'] = torch.stack(padded_labels)
            batch['attention_mask'] = torch.stack(padded_attention_masks)
        except Exception as e:
            print(f"Error stacking tensors: {e}")
            print(f"Shapes: input_ids={[x.shape for x in padded_input_ids[:3]]}")
            raise e
        
        # Final device check - ensure all tensors are on the correct device
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                if value.device != self.device:
                    print(f"Warning: Moving {key} from {value.device} to {self.device}")
                    batch[key] = value.to(self.device)
        
        return batch

def setup_training_args(output_dir="./checkpoints/sophiavl_lora", num_epochs=3, batch_size=1):
    """Setup training arguments optimized for LoRA fine-tuning"""
    
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,  # Smaller batch size for stability
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=16,  # Increase to maintain effective batch size
        warmup_steps=50,  # Reduced warmup steps
        learning_rate=5e-4,  # Higher LR for LoRA
        weight_decay=0.01,
        logging_dir=f"./logs/sophiavl_lora_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        logging_steps=10,
        save_steps=100,  # More frequent saves
        save_total_limit=3,
        eval_strategy="steps",  # Updated parameter name
        eval_steps=100,  # More frequent evaluation
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,  # Use mixed precision
        dataloader_pin_memory=False,  # Disable for memory efficiency
        remove_unused_columns=False,  # Keep all columns for vision models
        report_to=[],  # Disable external reporting to avoid issues
        seed=42,
        data_seed=42,
        group_by_length=False,  # Disable for vision models
        dataloader_num_workers=0,  # Single-threaded to avoid issues
        max_grad_norm=1.0,  # Gradient clipping for stability
    )

def train_sophiavl_lora():
    """Main training function for SophiaVL LoRA"""
    
    print("="*60)
    print("STARTING SOPHIAVL-R1 LORA TRAINING")
    print("="*60)
    
    # Get training data with validation
    training_data = get_training_data()
    if not training_data:
        raise ValueError("No training data available!")
    
    print(f"Training on {len(training_data)} examples from testmini split")
    
    # Create dataset with more robust error handling
    print("Creating dataset...")
    
    # Filter out problematic examples first
    print("Filtering training data...")
    filtered_data = []
    for i, example in enumerate(training_data):
        try:
            # Basic validation
            if (example.get('image') is not None and 
                example.get('formatted_question') and 
                len(example['formatted_question']) < 2000):  # Length limit
                filtered_data.append(example)
            else:
                print(f"Skipping example {i}: missing data or too long")
        except Exception as e:
            print(f"Error checking example {i}: {e}")
            continue
    
    print(f"Filtered data: {len(filtered_data)} examples (from {len(training_data)})")
    
    if len(filtered_data) < 10:
        raise ValueError("Too few valid training examples after filtering!")
    
    # Split data for validation (use 10% for validation)
    split_idx = int(0.9 * len(filtered_data))
    train_split = filtered_data[:split_idx]
    val_split = filtered_data[split_idx:] if split_idx < len(filtered_data) else filtered_data[-10:]  # At least 10 for validation
    
    print(f"Train split: {len(train_split)} examples")
    print(f"Validation split: {len(val_split)} examples")
    
    # Create datasets with reduced max_length
    train_dataset = MathVistaDataset(train_split, model_processor, max_length=768)
    val_dataset = MathVistaDataset(val_split, model_processor, max_length=768)
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")
    
    # Test the dataset and collator before training
    print("Testing data loading...")
    valid_items = []
    for i in range(min(10, len(train_dataset))):
        try:
            item = train_dataset[i]
            if item is not None and 'input_ids' in item:
                valid_items.append(item)
                if len(valid_items) >= 2:
                    break
        except Exception as e:
            print(f"Error loading item {i}: {e}")
            continue
    
    if len(valid_items) < 2:
        raise ValueError("Could not load enough valid training examples")
    
    print(f"✅ Loaded {len(valid_items)} valid items for testing")
    
    # Test collator with device fix
    data_collator = CustomDataCollator(model_processor)
    test_batch = data_collator(valid_items[:2])
    print(f"✅ Data collator working: {test_batch['input_ids'].shape}")
    print(f"✅ Data on correct device: {test_batch['input_ids'].device}")
    
    # Setup training arguments with more conservative settings
    training_args = setup_training_args(
        output_dir="./checkpoints/sophiavl_lora",
        num_epochs=1,  # Start with just 1 epoch
        batch_size=1   # Keep batch size small
    )
    
    # Create trainer
    print("Initializing trainer...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        tokenizer=model_processor.tokenizer,  # Pass tokenizer for saving
    )
    
    # Print model info before training
    print("\nModel configuration:")
    print(f"Model type: {type(model).__name__}")
    print(f"Model device: {next(model.parameters()).device}")
    if hasattr(model, 'print_trainable_parameters'):
        model.print_trainable_parameters()
    
    # Start training
    print("\nStarting training...")
    try:
        # Ensure device consistency before training
        ensure_device_consistency()
        
        # Clear memory before training
        clear_memory()
        
        # Train the model
        train_result = trainer.train()
        
        # Check if training result is valid
        if train_result is None:
            raise ValueError("Training returned None result")
        
        # Save the final model
        print("\nSaving trained model...")
        trainer.save_model()
        trainer.save_state()
        
        # Print training results
        print("\n" + "="*60)
        print("TRAINING COMPLETED SUCCESSFULLY!")
        print("="*60)
        print(f"Final training loss: {getattr(train_result, 'training_loss', 'N/A')}")
        
        if hasattr(train_result, 'metrics') and train_result.metrics:
            print(f"Training time: {train_result.metrics.get('train_runtime', 'N/A')} seconds")
            print(f"Samples per second: {train_result.metrics.get('train_samples_per_second', 'N/A')}")
            
            # Save training metrics
            with open(f"{training_args.output_dir}/training_metrics.json", "w") as f:
                json.dump(train_result.metrics, f, indent=2)
        
        return trainer, train_result
        
    except Exception as e:
        print(f"\nTraining failed with error: {e}")
        import traceback
        traceback.print_exc()
        
        print("Attempting to save checkpoint...")
        try:
            if 'trainer' in locals():
                trainer.save_model(f"{training_args.output_dir}/emergency_checkpoint")
                print("Emergency checkpoint saved!")
            else:
                print("Trainer not initialized, cannot save checkpoint")
        except Exception as save_error:
            print(f"Could not save emergency checkpoint: {save_error}")
        
        raise e

# Test a single example first to ensure everything works
def test_single_example():
    """Test processing a single example to debug any issues"""
    print("Testing single example processing...")
    
    training_data = get_training_data()
    if not training_data:
        print("No training data available!")
        return False
    
    try:
        # Test dataset creation
        test_dataset = MathVistaDataset([training_data[0]], model_processor)
        test_item = test_dataset[0]
        
        print("Single example processed successfully!")
        print(f"Input shape: {test_item['input_ids'].shape}")
        print(f"Input device: {test_item['input_ids'].device}")
        print(f"Labels shape: {test_item['labels'].shape}")
        print(f"Labels device: {test_item['labels'].device}")
        if 'pixel_values' in test_item:
            print(f"Pixel values shape: {test_item['pixel_values'].shape}")
            print(f"Pixel values device: {test_item['pixel_values'].device}")
        
        # Test data collator
        collator = CustomDataCollator(model_processor)
        batch = collator([test_item])
        
        print("Data collation successful!")
        print(f"Batch input_ids shape: {batch['input_ids'].shape}")
        print(f"Batch input_ids device: {batch['input_ids'].device}")
        print(f"Batch labels shape: {batch['labels'].shape}")
        print(f"Batch labels device: {batch['labels'].device}")
        print(f"Target device: {device}")
        
        return True
        
    except Exception as e:
        print(f"Error in single example test: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run the test first
if test_single_example():
    print("\n✅ Single example test passed! Ready to start training.")
    
    # Ask user confirmation before starting full training
    print("\n" + "="*60)
    print("READY TO START TRAINING")
    print("="*60)
    print("This will fine-tune the SophiaVL-R1 model with LoRA on the MathVista testmini dataset.")
    print("Training will take several hours depending on your hardware.")
    print("\nTo start training, run the next cell or call: train_sophiavl_lora()")
else:
    print("\n❌ Single example test failed. Please check the error messages above.")

# Store training function for easy access
training_functions = ['train_sophiavl_lora', 'test_single_example', 'setup_training_args']
print(f"\nAvailable training functions: {training_functions}")

Testing single example processing...
Pre-processing 1 examples...
Dataset initialized with 1 valid examples from 1 raw examples
Single example processed successfully!
Input shape: torch.Size([116])
Input device: cpu
Labels shape: torch.Size([116])
Labels device: cpu
Pixel values shape: torch.Size([5616, 1176])
Pixel values device: cpu
Data collation successful!
Batch input_ids shape: torch.Size([1, 116])
Batch input_ids device: cuda:1
Batch labels shape: torch.Size([1, 116])
Batch labels device: cuda:1
Target device: cuda:1

✅ Single example test passed! Ready to start training.

READY TO START TRAINING
This will fine-tune the SophiaVL-R1 model with LoRA on the MathVista testmini dataset.
Training will take several hours depending on your hardware.

To start training, run the next cell or call: train_sophiavl_lora()

Available training functions: ['train_sophiavl_lora', 'test_single_example', 'setup_training_args']


In [38]:
# Start Training SophiaVL-R1 LoRA
# Execute this cell to begin the fine-tuning process

print("🚀 Starting SophiaVL-R1 LoRA Fine-tuning...")
print("This may take several hours depending on your hardware.")
print("Monitor the progress in the logs and tensorboard.")

try:
    # Start the training
    trainer, results = train_sophiavl_lora()
    
    print("\n🎉 Training completed successfully!")
    print("✅ Model saved to ./checkpoints/sophiavl_lora/")
    print("✅ Logs saved to ./logs/")
    print("✅ Training metrics saved")
    
    # Clear memory after training
    clear_memory()
    
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
    print("Partial model may be saved in checkpoints directory")
    
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    print("Check the error messages above for debugging")
    
    # Clear memory even if training failed
    clear_memory()

🚀 Starting SophiaVL-R1 LoRA Fine-tuning...
This may take several hours depending on your hardware.
Monitor the progress in the logs and tensorboard.
STARTING SOPHIAVL-R1 LORA TRAINING
Training on 1000 examples from testmini split
Creating dataset...
Filtering training data...
Skipping example 17: missing data or too long
Skipping example 133: missing data or too long
Filtered data: 998 examples (from 1000)
Train split: 898 examples
Validation split: 100 examples
Pre-processing 898 examples...
Dataset initialized with 100 valid examples from 898 raw examples
Pre-processing 100 examples...
Dataset initialized with 100 valid examples from 898 raw examples
Pre-processing 100 examples...
Dataset initialized with 100 valid examples from 100 raw examples
Train dataset size: 100
Validation dataset size: 100
Testing data loading...
✅ Loaded 2 valid items for testing
✅ Data collator working: torch.Size([2, 116])
✅ Data on correct device: cuda:1
Initializing trainer...

Model configuration:
Model

/tmp/ipykernel_61219/1573315394.py:404: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Cleared memory. Current CPU memory usage: 19142.24 MB, GPU memory usage: 14552.92 MB

Training failed with error: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cuda:1 (when checking argument in method wrapper_CUDA__index_select)
Attempting to save checkpoint...


Traceback (most recent call last):
  File "/tmp/ipykernel_61219/1573315394.py", line 430, in train_sophiavl_lora
    train_result = trainer.train()
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/transformers/trainer.py", line 2328, in train
    return inner_training_loop(
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/transformers/trainer.py", line 2672, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/transformers/trainer.py", line 4009, in training_step
    loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/transformers/trainer.py", line 4099, in compute_loss
    outputs = model(**inputs)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torch/nn/m

Emergency checkpoint saved!

❌ Training failed: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cuda:1 (when checking argument in method wrapper_CUDA__index_select)
Check the error messages above for debugging
Cleared memory. Current CPU memory usage: 19142.49 MB, GPU memory usage: 14764.53 MB


In [37]:
# Additional Device Fix - Force all model components to correct device
print("🔧 Ensuring all model components are on the correct device...")

# Move the entire model to the correct device explicitly
model = model.to(device)

# For PEFT models, we need to ensure both base model and adapters are on the correct device
if hasattr(model, 'base_model'):
    model.base_model = model.base_model.to(device)

# Check embedding layer specifically (this is where the error occurs)
if hasattr(model, 'get_input_embeddings'):
    embeddings = model.get_input_embeddings()
    if embeddings is not None:
        embeddings = embeddings.to(device)
        print(f"✅ Input embeddings moved to: {embeddings.weight.device}")

# Force all parameters to be on the correct device
print("🔧 Moving all model parameters to the correct device...")
for name, param in model.named_parameters():
    if param.device != device:
        print(f"Moving parameter {name} from {param.device} to {device}")
        param.data = param.data.to(device)
        if param.grad is not None:
            param.grad = param.grad.to(device)

# Force all buffers to be on the correct device
print("🔧 Moving all model buffers to the correct device...")
for name, buffer in model.named_buffers():
    if buffer.device != device:
        print(f"Moving buffer {name} from {buffer.device} to {device}")
        buffer.data = buffer.data.to(device)

# Verify final device placement
print("🔍 Verifying model device placement...")
sample_param = next(model.parameters())
print(f"✅ Model parameters on: {sample_param.device}")

if hasattr(model, 'get_input_embeddings'):
    embeddings = model.get_input_embeddings()
    if embeddings is not None:
        print(f"✅ Input embeddings on: {embeddings.weight.device}")

print(f"✅ Target device: {device}")
print("🚀 Device fix completed! Ready to try training again.")

🔧 Ensuring all model components are on the correct device...
✅ Input embeddings moved to: cuda:1
🔧 Moving all model parameters to the correct device...
🔧 Moving all model buffers to the correct device...
🔍 Verifying model device placement...
✅ Model parameters on: cuda:1
✅ Input embeddings on: cuda:1
✅ Target device: cuda:1
🚀 Device fix completed! Ready to try training again.


In [12]:
# Test Updated Dataset and Collator - Fresh Version
# Run this cell to test the improved data processing before full training

print("🔧 Testing Updated Dataset and Data Collator (Fresh)...")

try:
    # Get training data
    training_data = get_training_data()
    if not training_data:
        print("❌ No training data available")
    else:
        print(f"✅ Got {len(training_data)} training examples")
        
        # Filter problematic examples like in the training function
        filtered_data = []
        for i, example in enumerate(training_data[:10]):  # Test first 10
            try:
                if (example.get('image') is not None and 
                    example.get('formatted_question') and 
                    len(example['formatted_question']) < 2000):
                    filtered_data.append(example)
                else:
                    print(f"Skipping example {i}: missing data or too long")
            except Exception as e:
                print(f"Error checking example {i}: {e}")
                continue
        
        print(f"Filtered {len(filtered_data)} valid examples from first 10")
        
        if len(filtered_data) < 2:
            print("❌ Not enough valid examples for testing")
        else:
            # Create fresh test dataset with the new implementation
            test_dataset = MathVistaDataset(filtered_data, model_processor, max_length=768)
            
            # Test individual items
            print("\nTesting individual items...")
            valid_items = []
            for i in range(min(3, len(test_dataset))):
                try:
                    item = test_dataset[i]
                    print(f"✅ Item {i}: input_ids={item['input_ids'].shape}, labels={item['labels'].shape}")
                    if 'pixel_values' in item:
                        print(f"   pixel_values={item['pixel_values'].shape}")
                    valid_items.append(item)
                except Exception as e:
                    print(f"❌ Item {i} failed: {e}")
            
            if len(valid_items) >= 2:
                # Create a fresh data collator instance
                print("\nTesting fresh data collator...")
                fresh_data_collator = CustomDataCollator(model_processor)
                
                # Test with 2 items that have different pixel_values shapes
                print("Testing with items that have different pixel_values shapes...")
                test_items = valid_items[:2]
                
                batch = fresh_data_collator(test_items)
                
                print(f"✅ Batch created successfully!")
                print(f"  Batch input_ids shape: {batch['input_ids'].shape}")
                print(f"  Batch labels shape: {batch['labels'].shape}")
                print(f"  Batch attention_mask shape: {batch['attention_mask'].shape}")
                
                if 'pixel_values' in batch:
                    print(f"  Batch pixel_values shape: {batch['pixel_values'].shape}")
                
                # Test with different batch sizes
                print("\nTesting different batch sizes...")
                for batch_size in [1, 2, 3]:
                    if len(valid_items) >= batch_size:
                        test_batch = fresh_data_collator(valid_items[:batch_size])
                        print(f"  Batch size {batch_size}: ✅ input_ids={test_batch['input_ids'].shape}, pixel_values={test_batch['pixel_values'].shape}")
                
                print("\n🎉 All tests passed! Data processing is working correctly.")
                print("The pixel_values padding is working properly.")
                print("You can now run the training with confidence.")
                
            else:
                print("❌ Not enough valid items for batch testing")
        
except Exception as e:
    print(f"❌ Testing failed: {e}")
    import traceback
    traceback.print_exc()
    print("Please check the error above before proceeding with training.")

🔧 Testing Updated Dataset and Data Collator (Fresh)...
✅ Got 1000 training examples
Filtered 10 valid examples from first 10
Pre-processing 10 examples...
Dataset initialized with 10 valid examples from 10 raw examples

Testing individual items...
✅ Item 0: input_ids=torch.Size([116]), labels=torch.Size([116])
   pixel_values=torch.Size([5616, 1176])
✅ Item 1: input_ids=torch.Size([74]), labels=torch.Size([74])
   pixel_values=torch.Size([3996, 1176])
✅ Item 2: input_ids=torch.Size([181]), labels=torch.Size([181])
   pixel_values=torch.Size([40, 1176])

Testing fresh data collator...
Testing with items that have different pixel_values shapes...
✅ Batch created successfully!
  Batch input_ids shape: torch.Size([2, 116])
  Batch labels shape: torch.Size([2, 116])
  Batch attention_mask shape: torch.Size([2, 116])
  Batch pixel_values shape: torch.Size([2, 5616, 1176])

Testing different batch sizes...
  Batch size 1: ✅ input_ids=torch.Size([1, 116]), pixel_values=torch.Size([1, 5616, 117

In [13]:
# Training Monitoring and Evaluation Functions
import glob

def monitor_training_progress(log_dir="./logs"):
    """Monitor training progress by reading log files"""
    import glob
    import re
    
    # Find the most recent log directory
    log_dirs = glob.glob(f"{log_dir}/sophiavl_lora_*")
    if not log_dirs:
        print("No training logs found!")
        return
    
    latest_log_dir = max(log_dirs, key=os.path.getctime)
    print(f"Monitoring logs in: {latest_log_dir}")
    
    # Try to read trainer_state.json for progress
    try:
        with open(f"{latest_log_dir}/trainer_state.json", "r") as f:
            state = json.load(f)
        
        print(f"Current epoch: {state.get('epoch', 'N/A')}")
        print(f"Global step: {state.get('global_step', 'N/A')}")
        print(f"Training loss: {state.get('train_loss', 'N/A')}")
        print(f"Learning rate: {state.get('learning_rate', 'N/A')}")
        
        if 'log_history' in state and state['log_history']:
            recent_logs = state['log_history'][-5:]  # Last 5 log entries
            print("\nRecent training logs:")
            for log in recent_logs:
                step = log.get('step', 'N/A')
                loss = log.get('train_loss', log.get('eval_loss', 'N/A'))
                lr = log.get('learning_rate', 'N/A')
                print(f"  Step {step}: Loss = {loss}, LR = {lr}")
                
    except FileNotFoundError:
        print("trainer_state.json not found. Training may not have started yet.")
    except Exception as e:
        print(f"Error reading training state: {e}")

def evaluate_trained_model(model_path="./checkpoints/sophiavl_lora", num_samples=50):
    """Evaluate the trained model on a subset of test data"""
    print(f"Loading trained model from {model_path}")
    
    try:
        from peft import PeftModel
        
        # Load the trained LoRA model
        trained_model = PeftModel.from_pretrained(model, model_path)
        trained_model.eval()
        
        # Get test data
        test_data = get_evaluation_data()[:num_samples]  # Limit for quick evaluation
        print(f"Evaluating on {len(test_data)} test samples...")
        
        correct_predictions = 0
        total_predictions = 0
        
        with torch.no_grad():
            for i, example in enumerate(tqdm(test_data, desc="Evaluating")):
                try:
                    # Prepare input
                    messages = [
                        {
                            "role": "user",
                            "content": [
                                {"type": "image", "image": example['image']},
                                {"type": "text", "text": example['formatted_question']}
                            ]
                        }
                    ]
                    
                    text = model_processor.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True
                    )
                    
                    inputs = model_processor(
                        text=[text],
                        images=[example['image']],
                        return_tensors="pt"
                    ).to(device)
                    
                    # Generate response
                    with torch.cuda.amp.autocast():
                        generated_ids = trained_model.generate(
                            **inputs,
                            max_new_tokens=100,
                            do_sample=False,
                            temperature=0.0,
                        )
                    
                    # Decode response
                    generated_text = model_processor.batch_decode(
                        generated_ids[:, inputs['input_ids'].shape[1]:], 
                        skip_special_tokens=True
                    )[0].strip()
                    
                    # Simple accuracy check (exact match)
                    ground_truth = str(example['ground_truth_answer']).strip()
                    prediction = generated_text.strip()
                    
                    if prediction.lower() == ground_truth.lower():
                        correct_predictions += 1
                    
                    total_predictions += 1
                    
                    # Print first few examples
                    if i < 5:
                        print(f"\nExample {i+1}:")
                        print(f"Question: {example['formatted_question'][:100]}...")
                        print(f"Ground Truth: {ground_truth}")
                        print(f"Prediction: {prediction}")
                        print(f"Correct: {'✅' if prediction.lower() == ground_truth.lower() else '❌'}")
                
                except Exception as e:
                    print(f"Error evaluating example {i}: {e}")
                    continue
        
        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        print(f"\n" + "="*50)
        print(f"EVALUATION RESULTS")
        print(f"="*50)
        print(f"Total samples evaluated: {total_predictions}")
        print(f"Correct predictions: {correct_predictions}")
        print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        return accuracy
        
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return None

def compare_before_after_training():
    """Compare model performance before and after training"""
    print("This function would compare base model vs fine-tuned model performance")
    print("Implementation requires loading both models and running evaluation")
    
# Utility function to check training status
def check_training_status():
    """Check if training is currently running or completed"""
    checkpoint_dirs = glob.glob("./checkpoints/sophiavl_lora*/")
    log_dirs = glob.glob("./logs/sophiavl_lora_*/")
    
    print(f"Found {len(checkpoint_dirs)} checkpoint directories")
    print(f"Found {len(log_dirs)} log directories")
    
    if checkpoint_dirs:
        latest_checkpoint = max(checkpoint_dirs, key=os.path.getctime)
        print(f"Latest checkpoint: {latest_checkpoint}")
        
        # Check if final model exists
        if os.path.exists(f"{latest_checkpoint}/pytorch_model.bin") or os.path.exists(f"{latest_checkpoint}/adapter_model.bin"):
            print("✅ Training appears to be completed!")
            return "completed"
        else:
            print("🔄 Training may be in progress...")
            return "in_progress"
    else:
        print("❌ No training checkpoints found")
        return "not_started"

print("Training monitoring functions loaded:")
print("- monitor_training_progress(): Check current training status")
print("- evaluate_trained_model(): Evaluate trained model on test data") 
print("- check_training_status(): Check if training is completed")
print("- compare_before_after_training(): Compare model performance")

Training monitoring functions loaded:
- monitor_training_progress(): Check current training status
- evaluate_trained_model(): Evaluate trained model on test data
- check_training_status(): Check if training is completed
- compare_before_after_training(): Compare model performance


In [19]:
# Evaluate the Trained SophiaVL-R1 Model
from peft import PeftModel

def evaluate_trained_sophiavl(model_path="./checkpoints/sophiavl_proper", num_samples=20):
    """Evaluate the trained SophiaVL-R1 model with proper image processing"""
    print("🔍 Evaluating trained SophiaVL-R1 model...")
    
    try:
        # Load the trained LoRA model
        print(f"📂 Loading model from {model_path}")
        trained_model = PeftModel.from_pretrained(model, model_path)
        trained_model = trained_model.to(device)  # Ensure model is on correct device
        trained_model.eval()
        
        # Get test data
        test_data = get_evaluation_data()[:num_samples]
        print(f"📊 Evaluating on {len(test_data)} test samples...")
        
        correct_predictions = 0
        total_predictions = 0
        results = []
        
        with torch.no_grad():
            for i, example in enumerate(tqdm(test_data, desc="Evaluating")):
                try:
                    # Prepare the conversation format
                    conversation = [
                        {
                            "role": "user",
                            "content": [
                                {"type": "image"},
                                {"type": "text", "text": example['formatted_question']}
                            ]
                        }
                    ]
                    
                    # Apply chat template
                    text = model_processor.apply_chat_template(
                        conversation, 
                        tokenize=False, 
                        add_generation_prompt=True
                    )
                    
                    # Process inputs
                    inputs = model_processor(
                        text=text,
                        images=example['image'],
                        return_tensors="pt"
                    )
                    
                    # Move all inputs to the correct device
                    for key in inputs:
                        if isinstance(inputs[key], torch.Tensor):
                            inputs[key] = inputs[key].to(device)
                    
                    # Generate response
                    with torch.cuda.amp.autocast():
                        generated_ids = trained_model.generate(
                            **inputs,
                            max_new_tokens=50,
                            do_sample=False,
                            temperature=0.0,
                            pad_token_id=model_processor.tokenizer.eos_token_id
                        )
                    
                    # Decode response
                    generated_text = model_processor.batch_decode(
                        generated_ids[:, inputs['input_ids'].shape[1]:], 
                        skip_special_tokens=True
                    )[0].strip()
                    
                    # Check accuracy
                    ground_truth = str(example['ground_truth_answer']).strip()
                    prediction = generated_text.strip()
                    
                    is_correct = prediction.lower() == ground_truth.lower()
                    if is_correct:
                        correct_predictions += 1
                    
                    total_predictions += 1
                    
                    # Store result
                    results.append({
                        'question_id': example.get('pid', f'test_{i}'),
                        'question': example['formatted_question'][:100] + "..." if len(example['formatted_question']) > 100 else example['formatted_question'],
                        'ground_truth': ground_truth,
                        'prediction': prediction,
                        'correct': is_correct
                    })
                    
                    # Print first few examples
                    if i < 5:
                        print(f"\n📝 Example {i+1}:")
                        print(f"   Question: {example['formatted_question'][:100]}...")
                        print(f"   Ground Truth: {ground_truth}")
                        print(f"   Prediction: {prediction}")
                        print(f"   {'✅ Correct' if is_correct else '❌ Incorrect'}")
                
                except Exception as e:
                    print(f"⚠️ Error evaluating example {i}: {e}")
                    continue
        
        # Calculate final accuracy
        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        
        print(f"\n{'='*50}")
        print(f"🎯 EVALUATION RESULTS")
        print(f"{'='*50}")
        print(f"📊 Total samples evaluated: {total_predictions}")
        print(f"✅ Correct predictions: {correct_predictions}")
        print(f"❌ Incorrect predictions: {total_predictions - correct_predictions}")
        print(f"🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        # Save results
        results_file = f"{model_path}/evaluation_results.json"
        with open(results_file, 'w') as f:
            json.dump({
                'accuracy': accuracy,
                'total_samples': total_predictions,
                'correct_predictions': correct_predictions,
                'results': results
            }, f, indent=2)
        
        print(f"💾 Results saved to: {results_file}")
        
        return accuracy, results
        
    except Exception as e:
        print(f"❌ Evaluation failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def compare_with_base_model(num_samples=10):
    """Compare trained model with base model performance"""
    print("⚖️ Comparing trained vs base model...")
    
    # Evaluate trained model
    print("1️⃣ Evaluating trained model...")
    trained_acc, trained_results = evaluate_trained_sophiavl(num_samples=num_samples)
    
    if trained_acc is None:
        print("❌ Could not evaluate trained model")
        return
    
    print(f"\n📈 COMPARISON RESULTS")
    print(f"{'='*40}")
    print(f"🎯 Trained Model Accuracy: {trained_acc:.4f} ({trained_acc*100:.2f}%)")
    print(f"📊 Sample Size: {num_samples}")
    
    return trained_acc, trained_results

print("🔍 Evaluation functions loaded!")
print("📝 Available functions:")
print("   - evaluate_trained_sophiavl(): Evaluate the trained model")
print("   - compare_with_base_model(): Compare trained vs base model")
print("")
print("💡 To evaluate the trained model:")
print("   accuracy, results = evaluate_trained_sophiavl(num_samples=20)")

🔍 Evaluation functions loaded!
📝 Available functions:
   - evaluate_trained_sophiavl(): Evaluate the trained model
   - compare_with_base_model(): Compare trained vs base model

💡 To evaluate the trained model:
   accuracy, results = evaluate_trained_sophiavl(num_samples=20)


In [20]:
# Test the Trained Model
print("🧪 Testing the trained SophiaVL-R1 model...")

try:
    accuracy, results = evaluate_trained_sophiavl(num_samples=15)
    
    if accuracy is not None:
        print(f"\n🎉 Evaluation completed!")
        print(f"🎯 Final accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    else:
        print("❌ Evaluation failed")
        
except Exception as e:
    print(f"❌ Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

🧪 Testing the trained SophiaVL-R1 model...
🔍 Evaluating trained SophiaVL-R1 model...
📂 Loading model from ./checkpoints/sophiavl_proper


/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.mod

📊 Evaluating on 15 test samples...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

/tmp/ipykernel_61219/45430311.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📝 Example 1:
   Question: In how many years, is the percentage of labor tax greater than 3 %?

Please provide your answer as a...
   Ground Truth: 
   Prediction: 2
   ❌ Incorrect

📝 Example 2:
   Question: Are there fewer small rubber objects that are in front of the large red metallic chopper than tiny c...
   Ground Truth: 
   Prediction: B
   ❌ Incorrect

📝 Example 3:
   Question: The players on a quiz show received the following scores. What is the mean of the numbers?'

Please ...
   Ground Truth: 
   Prediction: 8
   ❌ Incorrect

📝 Example 4:
   Question: Some friends played a trivia game and recorded their scores. What is the mode of the numbers?'

Plea...
   Ground Truth: 
   Prediction: 8
   ❌ Incorrect

📝 Example 3:
   Question: The players on a quiz show received the following scores. What is the mean of the numbers?'

Please ...
   Ground Truth: 
   Prediction: 8
   ❌ Incorrect

📝 Example 4:
   Question: Some friends played a trivia game and recorded their scores. What is

## Import Intern VL3